# Production Prediction Pipeline

## Objective

This notebook establishes the production-oriented inference layer of the customer churn prediction system.

The pipeline accepts new customer data and produces:

- Churn probability
- Churn prediction
- Business risk level
- Risk score
- Retention priority
- Expected revenue at risk
- Recommended business action

The inference pipeline uses the production artifacts created during model development:

1. Final XGBoost churn model
2. Saved preprocessing pipeline
3. Optimized classification threshold

The notebook is designed to ensure that inference is:

- Reproducible
- Consistent with model development
- Validated
- Business-oriented
- Suitable for downstream deployment

The resulting prediction engine will later be integrated into the Streamlit application.

In [1]:
# Import Libraries

import json
import sys
import warnings
from datetime import datetime
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

### Project Paths

In [2]:
# Project Paths

# Project root
PROJECT_ROOT = Path("..").resolve()

# Main directories
MODEL_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

# Prediction output directory
PREDICTION_DIR = OUTPUT_DIR / "predictions"

# Create required directories
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

# Production artifact paths
MODEL_PATH = MODEL_DIR / "final_churn_model.pkl"
PREPROCESSOR_PATH = MODEL_DIR / "preprocessor.pkl"
THRESHOLD_PATH = MODEL_DIR / "final_threshold.pkl"

# Output paths
PREDICTION_OUTPUT_PATH = (
    PREDICTION_DIR / "customer_churn_predictions.csv"
)

PRIORITY_OUTPUT_PATH = (
    PREDICTION_DIR / "top_retention_priority_customers.csv"
)

METADATA_OUTPUT_PATH = (
    MODEL_DIR / "production_model_metadata.json"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Model directory: {MODEL_DIR}")
print(f"Prediction directory: {PREDICTION_DIR}")

Project root: C:\Users\USER\Documents\Customer-Churn-System
Model directory: C:\Users\USER\Documents\Customer-Churn-System\models
Prediction directory: C:\Users\USER\Documents\Customer-Churn-System\outputs\predictions


## Validate Production Artifacts

In [3]:
artifact_paths = {
    "Model": MODEL_PATH,
    "Preprocessor": PREPROCESSOR_PATH,
    "Threshold": THRESHOLD_PATH
}

artifact_status = {}

for artifact_name, artifact_path in artifact_paths.items():
    exists = artifact_path.exists()

    artifact_status[artifact_name] = exists

    print(
        f"{artifact_name}: "
        f"{'✓ Found' if exists else '✗ Missing'}"
    )

if not all(artifact_status.values()):
    missing_artifacts = [
        name
        for name, exists in artifact_status.items()
        if not exists
    ]

    raise FileNotFoundError(
        "Missing production artifacts: "
        + ", ".join(missing_artifacts)
    )

print("\nAll production artifacts are available.")

Model: ✓ Found
Preprocessor: ✓ Found
Threshold: ✓ Found

All production artifacts are available.


## Load Production Artifacts

In [4]:
final_model = joblib.load(MODEL_PATH)

preprocessor = joblib.load(PREPROCESSOR_PATH)

final_threshold = float(
    joblib.load(THRESHOLD_PATH)
)

print("Production artifacts loaded successfully.")
print(f"Model: {MODEL_PATH}")
print(f"Preprocessor: {PREPROCESSOR_PATH}")
print(f"Classification threshold: {final_threshold:.4f}")

Production artifacts loaded successfully.
Model: C:\Users\USER\Documents\Customer-Churn-System\models\final_churn_model.pkl
Preprocessor: C:\Users\USER\Documents\Customer-Churn-System\models\preprocessor.pkl
Classification threshold: 0.3400


# Validate Classification Threshold

In [5]:
if not 0 < final_threshold < 1:
    raise ValueError(
        f"Invalid classification threshold: {final_threshold}. "
        "Threshold must be between 0 and 1."
    )

print(
    f"Valid production classification threshold: "
    f"{final_threshold:.4f}"
)

Valid production classification threshold: 0.3400


# Define Expected Raw Customer Schema

In [6]:
# Define Expected Raw Customer Schema

EXPECTED_RAW_COLUMNS = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "tenure",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges"
]

print(f"Expected customer input features: {len(EXPECTED_RAW_COLUMNS)}")

Expected customer input features: 19


In [7]:
# Feature Engineering

def engineer_features(df):
    """
    Apply the feature engineering required by the churn model.

    The transformation must remain consistent with the
    feature engineering used during model development.
    """

    data = df.copy()

    # Remove identifier if present

    if "customerID" in data.columns:
        data = data.drop(columns=["customerID"])

    # Convert TotalCharges to numeric

    if "TotalCharges" in data.columns:
        data["TotalCharges"] = pd.to_numeric(
            data["TotalCharges"],
            errors="coerce"
        )

    # Contract duration

    contract_mapping = {
        "Month-to-month": 1,
        "One year": 12,
        "Two year": 24
    }

    if "Contract" in data.columns:
        data["ContractMonths"] = (
            data["Contract"].map(contract_mapping)
        )

    # Total subscribed services

    service_columns = [
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies"
    ]

    available_services = [
        column
        for column in service_columns
        if column in data.columns
    ]

    if available_services:
        data["TotalServices"] = (
            data[available_services]
            .apply(
                lambda row: sum(
                    value == "Yes"
                    for value in row
                ),
                axis=1
            )
        )

    return data

# Input Validation

In [8]:
# Input Validation

def validate_customer_input(df):
    """
    Validate raw customer data before prediction.
    """

    if not isinstance(df, pd.DataFrame):
        raise TypeError(
            "Customer input must be a pandas DataFrame."
        )

    if df.empty:
        raise ValueError(
            "Customer input cannot be empty."
        )

    missing_columns = [
        column
        for column in EXPECTED_RAW_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            "Missing required customer features: "
            + ", ".join(missing_columns)
        )

    # Validate numeric features
    numeric_columns = [
        "SeniorCitizen",
        "tenure",
        "MonthlyCharges"
    ]

    for column in numeric_columns:

        if not pd.api.types.is_numeric_dtype(df[column]):
            raise TypeError(
                f"{column} must be numeric."
            )

    # Validate ranges
    if (df["tenure"] < 0).any():
        raise ValueError(
            "tenure cannot contain negative values."
        )

    if (df["MonthlyCharges"] < 0).any():
        raise ValueError(
            "MonthlyCharges cannot contain negative values."
        )

    if not df["SeniorCitizen"].isin([0, 1]).all():
        raise ValueError(
            "SeniorCitizen must contain only 0 or 1."
        )

    # Validate categorical values
    categorical_values = {
        "gender": {"Male", "Female"},
        "Partner": {"Yes", "No"},
        "Dependents": {"Yes", "No"},
        "PhoneService": {"Yes", "No"},
        "MultipleLines": {"Yes", "No", "No phone service"},
        "InternetService": {"DSL", "Fiber optic", "No"},
        "OnlineSecurity": {"Yes", "No", "No internet service"},
        "OnlineBackup": {"Yes", "No", "No internet service"},
        "DeviceProtection": {"Yes", "No", "No internet service"},
        "TechSupport": {"Yes", "No", "No internet service"},
        "StreamingTV": {"Yes", "No", "No internet service"},
        "StreamingMovies": {"Yes", "No", "No internet service"},
        "Contract": {
            "Month-to-month",
            "One year",
            "Two year"
        },
        "PaperlessBilling": {"Yes", "No"},
        "PaymentMethod": {
            "Electronic check",
            "Mailed check",
            "Bank transfer (automatic)",
            "Credit card (automatic)"
        }
    }

    for column, allowed_values in categorical_values.items():

        if column not in df.columns:
            continue

        invalid_values = set(
            df[column].dropna().unique()
        ) - allowed_values

        if invalid_values:
            raise ValueError(
                f"Invalid values found in {column}: "
                f"{invalid_values}"
            )

    return True

# Prediction Function

In [9]:
# Core Churn Prediction Function

def predict_churn(
    customer_data: pd.DataFrame,
    model,
    preprocessor,
    threshold: float
) -> pd.DataFrame:
    """
    Generate churn predictions for customer data.

    Returns a DataFrame containing:
    - Churn probability
    - Churn probability percentage
    - Churn prediction
    - Prediction label
    """

    # Validate raw input
    validate_customer_input(customer_data)

    # Apply the same feature engineering used during training
    data_engineered = engineer_features(customer_data)

    # Apply the saved preprocessing pipeline
    X_processed = preprocessor.transform(
        data_engineered
    )

    # Generate probability of churn
    churn_probability = (
        model.predict_proba(X_processed)[:, 1]
    )

    # Apply optimized production threshold
    churn_prediction = (
        churn_probability >= threshold
    ).astype(int)

    # Preserve original customer information
    results = customer_data.copy()

    results["Churn_Probability"] = churn_probability

    results["Churn_Probability_Percent"] = (
        churn_probability * 100
    )

    results["Churn_Prediction"] = churn_prediction

    results["Prediction_Label"] = np.where(
        churn_prediction == 1,
        "Churn",
        "No Churn"
    )

    return results

## Business Risk Segmentation

In [10]:
# Business Risk Segmentation

LOW_RISK_THRESHOLD = 0.40
HIGH_RISK_THRESHOLD = 0.70


def assign_risk_level(probability):
    """
    Convert churn probability into business risk level.

    Note:
    These business risk thresholds are separate from
    the machine-learning classification threshold.
    """

    if probability >= HIGH_RISK_THRESHOLD:
        return "High Risk"

    if probability >= LOW_RISK_THRESHOLD:
        return "Medium Risk"

    return "Low Risk"


def add_risk_level(prediction_df):
    """
    Add business risk classification.
    """

    results = prediction_df.copy()

    results["Risk_Level"] = (
        results["Churn_Probability"]
        .apply(assign_risk_level)
    )

    return results

## Classification Threshold vs Business Risk Threshold

The system uses two different threshold concepts.

### Machine Learning Classification Threshold

The optimized model threshold determines whether the model predicts:

- `Churn`
- `No Churn`

The current production threshold is loaded from the saved artifact.

### Business Risk Thresholds

Business risk segmentation is independent of the classification threshold:

- **Low Risk:** probability < 0.40
- **Medium Risk:** 0.40 ≤ probability < 0.70
- **High Risk:** probability ≥ 0.70

This distinction allows the business to prioritize customers based on severity while preserving the optimized machine-learning decision threshold.

## Retention Priority + Revenue Risk

In [11]:
# Retention Priority and Revenue-at-Risk Metrics

RISK_SCORE_MAPPING = {
    "Low Risk": 1,
    "Medium Risk": 2,
    "High Risk": 3
}


def add_retention_priority(prediction_df):
    """
    Add business prioritization and revenue-at-risk metrics.

    Retention_Priority_Score is a heuristic:
        Risk Score × Monthly Charges

    Expected revenue-at-risk metrics incorporate the model's
    estimated churn probability.
    """

    results = prediction_df.copy()

    # Risk score

    results["Risk_Score"] = (
        results["Risk_Level"]
        .map(RISK_SCORE_MAPPING)
    )

    # Business prioritization heuristic

    results["Retention_Priority_Score"] = (
        results["Risk_Score"]
        * results["MonthlyCharges"]
    )

    # Expected revenue at risk

    results["Expected_Monthly_Revenue_at_Risk"] = (
        results["Churn_Probability"]
        * results["MonthlyCharges"]
    )

    results["Expected_Annual_Revenue_at_Risk"] = (
        results["Expected_Monthly_Revenue_at_Risk"]
        * 12
    )

    return results

## Business Recommendation Engine

In [12]:
# Business Recommendation Engine

RETENTION_STRATEGY = {
    "High Risk": (
        "Immediate retention outreach, personalized offers, "
        "loyalty incentives, and proactive customer support."
    ),

    "Medium Risk": (
        "Monitor customer engagement, send personalized "
        "communication, and offer targeted promotions."
    ),

    "Low Risk": (
        "Maintain customer relationship through loyalty "
        "programs, regular engagement, and service improvements."
    )
}


def add_recommended_action(prediction_df):
    """
    Add recommended business action based on risk level.
    """

    results = prediction_df.copy()

    results["Recommended_Action"] = (
        results["Risk_Level"]
        .map(RETENTION_STRATEGY)
    )

    return results

# Complete Production Prediction Pipeline

In [13]:
def run_prediction_pipeline(
    customer_data,
    model=final_model,
    preprocessor=preprocessor,
    threshold=final_threshold
):
    """
    Execute the complete customer churn inference workflow.

    Pipeline:

    Raw customer data
        ↓
    Input validation
        ↓
    Feature engineering
        ↓
    Preprocessing
        ↓
    Churn probability
        ↓
    Classification
        ↓
    Business risk
        ↓
    Retention priority
        ↓
    Revenue at risk
        ↓
    Recommended action
    """

    # Model predictions
    results = predict_churn(
        customer_data=customer_data,
        model=model,
        preprocessor=preprocessor,
        threshold=threshold
    )

    # Business risk
    results = add_risk_level(results)

    # Retention and financial prioritization
    results = add_retention_priority(results)

    # Business recommendation
    results = add_recommended_action(results)

    return results

## Create new customer data

In [14]:
# Create Sample New Customers 

new_customers = pd.DataFrame({
    "gender": ["Male", "Female", "Male"],
    "SeniorCitizen": [1, 0, 0],
    "Partner": ["No", "Yes", "No"],
    "Dependents": ["No", "Yes", "No"],
    "tenure": [5, 48, 8],

    "PhoneService": ["Yes", "Yes", "Yes"],
    "MultipleLines": ["Yes", "No", "Yes"],

    "InternetService": [
        "Fiber optic",
        "DSL",
        "Fiber optic"
    ],

    "OnlineSecurity": ["No", "Yes", "No"],
    "OnlineBackup": ["No", "Yes", "No"],
    "DeviceProtection": ["No", "Yes", "No"],
    "TechSupport": ["No", "Yes", "No"],
    "StreamingTV": ["Yes", "No", "Yes"],
    "StreamingMovies": ["Yes", "No", "Yes"],

    "Contract": [
        "Month-to-month",
        "One year",
        "Month-to-month"
    ],

    "PaperlessBilling": ["Yes", "No", "Yes"],

    "PaymentMethod": [
        "Electronic check",
        "Credit card (automatic)",
        "Electronic check"
    ],

    "MonthlyCharges": [
        95.50,
        65.25,
        101.20
    ],

    "TotalCharges": [
        477.5,
        3132.0,
        809.6
    ]
})

new_customers

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Male,1,No,No,5,Yes,Yes,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,95.50,477.5
1,Female,0,Yes,Yes,48,Yes,No,DSL,Yes,Yes,Yes,Yes,No,No,One year,No,Credit card (automatic),65.25,3132.0
2,Male,0,No,No,8,Yes,Yes,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,101.20,809.6


## End-to-End Prediction

In [15]:
# Run End-to-End Prediction

prediction_results = run_prediction_pipeline(
    customer_data=new_customers
)

prediction_results

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,Churn_Probability,Churn_Probability_Percent,Churn_Prediction,Prediction_Label,Risk_Level,Risk_Score,Retention_Priority_Score,Expected_Monthly_Revenue_at_Risk,Expected_Annual_Revenue_at_Risk,Recommended_Action
0,Male,1,No,No,5,Yes,Yes,Fiber optic,No,No,...,0.855289,85.528877,1,Churn,High Risk,3,286.50,81.680075,980.160901,"Immediate retention outreach, personalized off..."
1,Female,0,Yes,Yes,48,Yes,No,DSL,Yes,Yes,...,0.033666,3.366644,0,No Churn,Low Risk,1,65.25,2.196735,26.360819,Maintain customer relationship through loyalty...
2,Male,0,No,No,8,Yes,Yes,Fiber optic,No,No,...,0.849059,84.905945,1,Churn,High Risk,3,303.60,85.924818,1031.097811,"Immediate retention outreach, personalized off..."


# Business-Friendly Output

In [16]:
# Business-Friendly Prediction Output

business_columns = [
    "Contract",
    "tenure",
    "MonthlyCharges",
    "InternetService",
    "PaymentMethod",
    "Churn_Probability",
    "Churn_Probability_Percent",
    "Prediction_Label",
    "Risk_Level",
    "Retention_Priority_Score",
    "Expected_Monthly_Revenue_at_Risk",
    "Expected_Annual_Revenue_at_Risk",
    "Recommended_Action"
]

business_output = (
    prediction_results[business_columns]
    .copy()
)

business_output

,Contract,tenure,MonthlyCharges,InternetService,PaymentMethod,Churn_Probability,Churn_Probability_Percent,Prediction_Label,Risk_Level,Retention_Priority_Score,Expected_Monthly_Revenue_at_Risk,Expected_Annual_Revenue_at_Risk,Recommended_Action
0,Month-to-month,5,95.50,Fiber optic,Electronic check,0.855289,85.528877,Churn,High Risk,286.50,81.680075,980.160901,"Immediate retention outreach, personalized off..."
1,One year,48,65.25,DSL,Credit card (automatic),0.033666,3.366644,No Churn,Low Risk,65.25,2.196735,26.360819,Maintain customer relationship through loyalty...
2,Month-to-month,8,101.20,Fiber optic,Electronic check,0.849059,84.905945,Churn,High Risk,303.60,85.924818,1031.097811,"Immediate retention outreach, personalized off..."


# Format Business Output

In [17]:
formatted_business_output = business_output.copy()

formatted_business_output[
    "Churn_Probability"
] = formatted_business_output[
    "Churn_Probability"
].round(4)

formatted_business_output[
    "Churn_Probability_Percent"
] = formatted_business_output[
    "Churn_Probability_Percent"
].round(2)

formatted_business_output[
    "Retention_Priority_Score"
] = formatted_business_output[
    "Retention_Priority_Score"
].round(2)

formatted_business_output[
    "Expected_Monthly_Revenue_at_Risk"
] = formatted_business_output[
    "Expected_Monthly_Revenue_at_Risk"
].round(2)

formatted_business_output[
    "Expected_Annual_Revenue_at_Risk"
] = formatted_business_output[
    "Expected_Annual_Revenue_at_Risk"
].round(2)

formatted_business_output

,Contract,tenure,MonthlyCharges,InternetService,PaymentMethod,Churn_Probability,Churn_Probability_Percent,Prediction_Label,Risk_Level,Retention_Priority_Score,Expected_Monthly_Revenue_at_Risk,Expected_Annual_Revenue_at_Risk,Recommended_Action
0,Month-to-month,5,95.50,Fiber optic,Electronic check,0.8553,85.529999,Churn,High Risk,286.50,81.68,980.16,"Immediate retention outreach, personalized off..."
1,One year,48,65.25,DSL,Credit card (automatic),0.0337,3.370000,No Churn,Low Risk,65.25,2.20,26.36,Maintain customer relationship through loyalty...
2,Month-to-month,8,101.20,Fiber optic,Electronic check,0.8491,84.910004,Churn,High Risk,303.60,85.92,1031.10,"Immediate retention outreach, personalized off..."


## Validate Individual Predictions

In [18]:
# Validate Individual Prediction Pipeline

assert len(prediction_results) == len(new_customers)

assert (
    prediction_results["Churn_Probability"]
    .between(0, 1)
    .all()
)

assert (
    prediction_results["Churn_Probability_Percent"]
    .between(0, 100)
    .all()
)

assert (
    prediction_results["Prediction_Label"]
    .isin(["Churn", "No Churn"])
    .all()
)

assert (
    prediction_results["Risk_Level"]
    .isin(
        [
            "Low Risk",
            "Medium Risk",
            "High Risk"
        ]
    )
    .all()
)

print("Individual prediction validation passed.")

Individual prediction validation passed.


## Validate Classification Threshold

In [19]:
# Validate Classification Threshold

expected_predictions = (
    prediction_results["Churn_Probability"]
    >= final_threshold
).astype(int)

assert (
    expected_predictions
    == prediction_results["Churn_Prediction"]
).all()

print(
    "Classification threshold validation passed."
)

print(
    f"Production threshold: {final_threshold:.4f}"
)

Classification threshold validation passed.
Production threshold: 0.3400


## Validate Business Risk Logic

In [20]:
# Validate Business Risk Logic

for _, row in prediction_results.iterrows():

    probability = row["Churn_Probability"]
    risk = row["Risk_Level"]

    if probability >= HIGH_RISK_THRESHOLD:
        assert risk == "High Risk"

    elif probability >= LOW_RISK_THRESHOLD:
        assert risk == "Medium Risk"

    else:
        assert risk == "Low Risk"

print("Business risk segmentation validation passed.")

Business risk segmentation validation passed.


## Batch Inference Dataset

In [21]:
# Load Batch Inference Dataset

CLEAN_DATA_PATH = (
    DATA_DIR / "cleaned" / "churn_clean.csv"
)

customer_dataset = pd.read_csv(
    CLEAN_DATA_PATH
)

print(
    f"Loaded {len(customer_dataset):,} customers "
    "for batch inference validation."
)

Loaded 7,043 customers for batch inference validation.


## Remove Target From Inference Data

In [22]:
# Prepare Batch Inference Data

batch_input = customer_dataset.copy()

# Target variable must never be used as an inference feature
if "Churn" in batch_input.columns:
    batch_input = batch_input.drop(
        columns=["Churn"]
    )

batch_input.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


## Run Batch Predictions

In [23]:
# Run Batch Inference

batch_predictions = run_prediction_pipeline(
    customer_data=batch_input
)

print(
    f"Batch inference completed for "
    f"{len(batch_predictions):,} customers."
)

batch_predictions.head()

Batch inference completed for 7,043 customers.


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Churn_Probability,Churn_Probability_Percent,Churn_Prediction,Prediction_Label,Risk_Level,Risk_Score,Retention_Priority_Score,Expected_Monthly_Revenue_at_Risk,Expected_Annual_Revenue_at_Risk,Recommended_Action
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,0.683173,68.317291,1,Churn,Medium Risk,2,59.70,20.392712,244.712548,"Monitor customer engagement, send personalized..."
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,0.035567,3.556687,0,No Churn,Low Risk,1,56.95,2.025533,24.306402,Maintain customer relationship through loyalty...
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,0.355927,35.592693,1,Churn,Low Risk,1,53.85,19.166665,229.999983,Maintain customer relationship through loyalty...
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,0.034066,3.406573,0,No Churn,Low Risk,1,42.30,1.440980,17.291765,Maintain customer relationship through loyalty...
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,0.648511,64.851097,1,Churn,Medium Risk,2,141.40,45.849727,550.196726,"Monitor customer engagement, send personalized..."


## Batch Risk Distribution

In [24]:
# Batch Risk Distribution

risk_distribution = (
    batch_predictions["Risk_Level"]
    .value_counts()
    .rename_axis("Risk_Level")
    .reset_index(name="Customer_Count")
)

risk_distribution["Percentage"] = (
    risk_distribution["Customer_Count"]
    / len(batch_predictions)
    * 100
).round(2)

risk_distribution

,Risk_Level,Customer_Count,Percentage
0,Low Risk,5057,71.80
1,Medium Risk,1388,19.71
2,High Risk,598,8.49


## Batch Summary

In [25]:
# Batch Prediction Summary

batch_summary = pd.DataFrame({
    "Metric": [
        "Total Customers",
        "Predicted Churn",
        "Predicted No Churn",
        "High Risk Customers",
        "Medium Risk Customers",
        "Low Risk Customers",
        "Total Expected Monthly Revenue at Risk",
        "Total Expected Annual Revenue at Risk"
    ],

    "Value": [
        len(batch_predictions),

        (
            batch_predictions["Churn_Prediction"] == 1
        ).sum(),

        (
            batch_predictions["Churn_Prediction"] == 0
        ).sum(),

        (
            batch_predictions["Risk_Level"] == "High Risk"
        ).sum(),

        (
            batch_predictions["Risk_Level"] == "Medium Risk"
        ).sum(),

        (
            batch_predictions["Risk_Level"] == "Low Risk"
        ).sum(),

        batch_predictions[
            "Expected_Monthly_Revenue_at_Risk"
        ].sum(),

        batch_predictions[
            "Expected_Annual_Revenue_at_Risk"
        ].sum()
    ]
})

batch_summary

,Metric,Value
0,Total Customers,7.043000e+03
1,Predicted Churn,2.339000e+03
2,Predicted No Churn,4.704000e+03
3,High Risk Customers,5.980000e+02
4,Medium Risk Customers,1.388000e+03
5,Low Risk Customers,5.057000e+03
6,Total Expected Monthly Revenue at Risk,1.398092e+05
7,Total Expected Annual Revenue at Risk,1.677710e+06


## Identify Highest Priority Customers

In [26]:
# Identify Highest Priority Customers

top_priority_customers = (
    batch_predictions
    .sort_values(
        [
            "Expected_Annual_Revenue_at_Risk",
            "Retention_Priority_Score"
        ],
        ascending=False
    )
    .head(20)
    .copy()
)

top_priority_customers

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Churn_Probability,Churn_Probability_Percent,Churn_Prediction,Prediction_Label,Risk_Level,Risk_Score,Retention_Priority_Score,Expected_Monthly_Revenue_at_Risk,Expected_Annual_Revenue_at_Risk,Recommended_Action
2208,7216-EWTRS,Female,1,Yes,No,1,Yes,Yes,Fiber optic,No,...,0.907717,90.771713,1,Churn,High Risk,3,302.40,91.497885,1097.974615,"Immediate retention outreach, personalized off..."
6482,5419-JPRRN,Male,0,No,No,1,Yes,Yes,Fiber optic,No,...,0.894049,89.404922,1,Churn,High Risk,3,304.35,90.701294,1088.415529,"Immediate retention outreach, personalized off..."
1704,0107-YHINA,Male,0,No,Yes,1,Yes,Yes,Fiber optic,No,...,0.889036,88.903557,1,Churn,High Risk,3,299.25,88.681299,1064.175592,"Immediate retention outreach, personalized off..."
6894,1400-MMYXY,Male,1,Yes,No,3,Yes,Yes,Fiber optic,No,...,0.837279,83.727867,1,Churn,High Risk,3,317.70,88.667811,1064.013726,"Immediate retention outreach, personalized off..."
4826,3389-YGYAI,Female,1,No,No,8,Yes,Yes,Fiber optic,No,...,0.835232,83.523216,1,Churn,High Risk,3,316.50,88.116991,1057.403888,"Immediate retention outreach, personalized off..."
5933,6496-SLWHQ,Male,1,No,No,3,Yes,Yes,Fiber optic,No,...,0.837279,83.727867,1,Churn,High Risk,3,315.00,87.914260,1054.971117,"Immediate retention outreach, personalized off..."
3956,4587-VVTOX,Female,0,Yes,No,6,Yes,Yes,Fiber optic,No,...,0.832623,83.262268,1,Churn,High Risk,3,315.90,87.675165,1052.101977,"Immediate retention outreach, personalized off..."
4459,3178-FESZO,Female,0,No,No,1,Yes,Yes,Fiber optic,No,...,0.870365,87.036476,1,Churn,High Risk,3,300.75,87.254070,1047.048837,"Immediate retention outreach, personalized off..."
6365,8884-ADFVN,Male,1,Yes,No,7,Yes,Yes,Fiber optic,No,...,0.854861,85.486061,1,Churn,High Risk,3,305.85,87.153039,1045.836463,"Immediate retention outreach, personalized off..."
6232,9681-OXGVC,Female,0,No,No,5,Yes,Yes,Fiber optic,No,...,0.867077,86.707703,1,Churn,High Risk,3,301.50,87.141244,1045.694925,"Immediate retention outreach, personalized off..."


## Validate Batch Processing

In [27]:
# Validate Batch Processing

assert len(batch_predictions) == len(batch_input)

assert (
    batch_predictions["Churn_Probability"]
    .between(0, 1)
    .all()
)

assert (
    batch_predictions["Churn_Probability_Percent"]
    .between(0, 100)
    .all()
)

assert (
    batch_predictions["Risk_Level"]
    .notna()
    .all()
)

assert (
    batch_predictions["Recommended_Action"]
    .notna()
    .all()
)

assert (
    batch_predictions[
        "Expected_Monthly_Revenue_at_Risk"
    ].notna()
    .all()
)

print("Batch prediction validation passed.")

Batch prediction validation passed.


## Save Predictions

In [28]:
# Save Batch Predictions

batch_predictions.to_csv(
    PREDICTION_OUTPUT_PATH,
    index=False
)

print(
    f"Predictions saved to:\n"
    f"{PREDICTION_OUTPUT_PATH}"
)

Predictions saved to:
C:\Users\USER\Documents\Customer-Churn-System\outputs\predictions\customer_churn_predictions.csv


## Save Top Priority Customers

In [29]:
# Save Top Priority Customers

top_priority_customers.to_csv(
    PRIORITY_OUTPUT_PATH,
    index=False
)

print(
    f"Priority customers saved to:\n"
    f"{PRIORITY_OUTPUT_PATH}"
)

Priority customers saved to:
C:\Users\USER\Documents\Customer-Churn-System\outputs\predictions\top_retention_priority_customers.csv


## Package Version Information

This makes the project more reproducible.

In [30]:
# Package Version Helper

def get_package_version(package_name):
    """
    Return installed package version.
    """

    try:
        return version(package_name)

    except PackageNotFoundError:
        return "Not installed"


package_versions = {
    "python": sys.version.split()[0],
    "pandas": get_package_version("pandas"),
    "numpy": get_package_version("numpy"),
    "scikit-learn": get_package_version("scikit-learn"),
    "xgboost": get_package_version("xgboost"),
    "joblib": get_package_version("joblib")
}

package_versions

{'python': '3.12.12',
 'pandas': '2.3.3',
 'numpy': '2.3.5',
 'scikit-learn': '1.6.0',
 'xgboost': '3.3.0',
 'joblib': '1.4.2'}

## Production Metadata

In [31]:
# Production Model Metadata

model_metadata = {
    "model_name": "Customer Churn XGBoost Production Model",
    "model_file": MODEL_PATH.name,
    "preprocessor_file": PREPROCESSOR_PATH.name,
    "threshold_file": THRESHOLD_PATH.name,

    "classification_threshold": final_threshold,

    "business_risk_thresholds": {
        "low_risk_upper_bound": LOW_RISK_THRESHOLD,
        "high_risk_lower_bound": HIGH_RISK_THRESHOLD
    },

    "model_type": type(final_model).__name__,
    "preprocessor_type": type(preprocessor).__name__,

    "package_versions": package_versions,

    "inference_timestamp": datetime.now().isoformat(),

    "input_feature_count": len(
        EXPECTED_RAW_COLUMNS
    ),

    "expected_raw_features": EXPECTED_RAW_COLUMNS
}

with open(
    METADATA_OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        model_metadata,
        file,
        indent=4
    )

print(
    f"Model metadata saved to:\n"
    f"{METADATA_OUTPUT_PATH}"
)

Model metadata saved to:
C:\Users\USER\Documents\Customer-Churn-System\models\production_model_metadata.json


## Production Readiness Check

In [32]:
# Production Readiness Check

production_checks = {
    "Model available": MODEL_PATH.exists(),

    "Preprocessor available":
        PREPROCESSOR_PATH.exists(),

    "Threshold available":
        THRESHOLD_PATH.exists(),

    "Valid threshold":
        0 < final_threshold < 1,

    "Prediction function available":
        callable(predict_churn),

    "Feature engineering available":
        callable(engineer_features),

    "Input validation available":
        callable(validate_customer_input),

    "Risk segmentation available":
        callable(add_risk_level),

    "Priority scoring available":
        callable(add_retention_priority),

    "Recommendation engine available":
        callable(add_recommended_action),

    "Complete pipeline available":
        callable(run_prediction_pipeline),

    "Individual prediction successful":
        len(prediction_results) > 0,

    "Batch prediction successful":
        len(batch_predictions) > 0,

    "Prediction output saved":
        PREDICTION_OUTPUT_PATH.exists(),

    "Priority output saved":
        PRIORITY_OUTPUT_PATH.exists(),

    "Metadata saved":
        METADATA_OUTPUT_PATH.exists()
}

production_status = pd.DataFrame(
    production_checks.items(),
    columns=["Check", "Status"]
)

production_status

,Check,Status
0,Model available,True
1,Preprocessor available,True
2,Threshold available,True
3,Valid threshold,True
4,Prediction function available,True
5,Feature engineering available,True
6,Input validation available,True
7,Risk segmentation available,True
8,Priority scoring available,True
9,Recommendation engine available,True


## Assert Production Readiness

In [33]:
# Assert Production Readiness

if not production_status["Status"].all():

    failed_checks = production_status.loc[
        ~production_status["Status"],
        "Check"
    ].tolist()

    raise RuntimeError(
        "Production readiness checks failed:\n"
        + "\n".join(
            f"- {check}"
            for check in failed_checks
        )
    )

print(
    "All production readiness checks passed."
)

All production readiness checks passed.


# Final Summary

Notebook 09 establishes the production-oriented inference layer of the customer churn prediction system.

The completed pipeline can:

1. Load the production XGBoost model.
2. Load the saved preprocessing pipeline.
3. Load the optimized classification threshold.
4. Validate incoming customer data.
5. Apply production feature engineering.
6. Transform customer features using the saved preprocessing pipeline.
7. Generate churn probabilities.
8. Apply the optimized classification threshold.
9. Generate churn predictions.
10. Segment customers into Low, Medium, and High Risk.
11. Calculate retention priority.
12. Estimate expected monthly revenue at risk.
13. Estimate expected annual revenue at risk.
14. Generate recommended retention actions.
15. Process individual customers.
16. Process complete customer datasets.
17. Validate individual predictions.
18. Validate batch predictions.
19. Export prediction results.
20. Export high-priority retention customers.
21. Store production model metadata.
22. Perform production-readiness checks.

## Important Architectural Principle

Notebook 09 is the validation and demonstration layer for the production inference system.

The reusable prediction logic should subsequently be extracted into a Python module and imported by the deployment application.

The next stage is therefore:

**Notebook 09 → Production Python Module → Streamlit Application**

In [34]:
from src.prediction_pipeline import (
    load_production_artifacts,
    run_prediction_pipeline
)

print("Prediction pipeline imported successfully.")

Prediction pipeline imported successfully.


In [35]:
import src

print("src imported successfully!")
print(src.__file__)

src imported successfully!
C:\Users\USER\Documents\Customer-Churn-System\src\__init__.py


In [36]:
import pandas as pd

new_customers = pd.DataFrame({
    "gender": ["Male", "Female", "Male"],
    "SeniorCitizen": [1, 0, 0],
    "Partner": ["No", "Yes", "No"],
    "Dependents": ["No", "Yes", "No"],
    "tenure": [5, 48, 8],
    "PhoneService": ["Yes", "Yes", "Yes"],
    "MultipleLines": ["Yes", "No", "Yes"],
    "InternetService": [
        "Fiber optic",
        "DSL",
        "Fiber optic"
    ],
    "OnlineSecurity": ["No", "Yes", "No"],
    "OnlineBackup": ["No", "Yes", "No"],
    "DeviceProtection": ["No", "Yes", "No"],
    "TechSupport": ["No", "Yes", "No"],
    "StreamingTV": ["Yes", "No", "Yes"],
    "StreamingMovies": ["Yes", "No", "Yes"],
    "Contract": [
        "Month-to-month",
        "One year",
        "Month-to-month"
    ],
    "PaperlessBilling": ["Yes", "No", "Yes"],
    "PaymentMethod": [
        "Electronic check",
        "Credit card (automatic)",
        "Electronic check"
    ],
    "MonthlyCharges": [
        95.50,
        65.25,
        101.20
    ],
    "TotalCharges": [
        477.5,
        3132.0,
        809.6
    ]
})

In [37]:
results = run_prediction_pipeline(
    new_customers
)

results

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,Churn_Probability,Churn_Probability_Percent,Churn_Prediction,Prediction_Label,Risk_Level,Risk_Score,Retention_Priority_Score,Expected_Monthly_Revenue_at_Risk,Expected_Annual_Revenue_at_Risk,Recommended_Action
0,Male,1,No,No,5,Yes,Yes,Fiber optic,No,No,...,0.855289,85.528877,1,Churn,High Risk,3,286.50,81.680075,980.160901,"Immediate retention outreach, personalized off..."
1,Female,0,Yes,Yes,48,Yes,No,DSL,Yes,Yes,...,0.033666,3.366644,0,No Churn,Low Risk,1,65.25,2.196735,26.360819,Maintain customer relationship through loyalty...
2,Male,0,No,No,8,Yes,Yes,Fiber optic,No,No,...,0.849059,84.905945,1,Churn,High Risk,3,303.60,85.924818,1031.097811,"Immediate retention outreach, personalized off..."


In [38]:
# Module Validation

assert len(results) == len(new_customers)

assert results["Churn_Probability"].between(
    0, 1
).all()

assert results["Churn_Probability_Percent"].between(
    0, 100
).all()

assert results["Churn_Prediction"].isin(
    [0, 1]
).all()

assert results["Prediction_Label"].isin(
    ["Churn", "No Churn"]
).all()

assert results["Risk_Level"].isin(
    [
        "Low Risk",
        "Medium Risk",
        "High Risk"
    ]
).all()

assert results[
    "Expected_Monthly_Revenue_at_Risk"
].notna().all()

assert results[
    "Expected_Annual_Revenue_at_Risk"
].notna().all()

assert results[
    "Recommended_Action"
].notna().all()

print("✓ Prediction module validation passed.")

✓ Prediction module validation passed.


In [39]:
model, preprocessor, threshold = (
    load_production_artifacts()
)

expected_predictions = (
    results["Churn_Probability"]
    >= threshold
).astype(int)

assert (
    results["Churn_Prediction"]
    == expected_predictions
).all()

print(
    f"✓ Classification threshold validated: "
    f"{threshold:.4f}"
)

✓ Classification threshold validated: 0.3400


In [40]:
customer_dataset = pd.read_csv(
    "../data/cleaned/churn_clean.csv"
)

batch_input = customer_dataset.drop(
    columns=["Churn"],
    errors="ignore"
)

batch_results = run_prediction_pipeline(
    batch_input
)

print(
    f"✓ Batch inference completed: "
    f"{len(batch_results):,} customers"
)

batch_results.head()

✓ Batch inference completed: 7,043 customers


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Churn_Probability,Churn_Probability_Percent,Churn_Prediction,Prediction_Label,Risk_Level,Risk_Score,Retention_Priority_Score,Expected_Monthly_Revenue_at_Risk,Expected_Annual_Revenue_at_Risk,Recommended_Action
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,0.683173,68.317291,1,Churn,Medium Risk,2,59.70,20.392712,244.712548,"Monitor customer engagement, send personalized..."
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,0.035567,3.556687,0,No Churn,Low Risk,1,56.95,2.025533,24.306402,Maintain customer relationship through loyalty...
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,0.355927,35.592693,1,Churn,Low Risk,1,53.85,19.166665,229.999983,Maintain customer relationship through loyalty...
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,0.034066,3.406573,0,No Churn,Low Risk,1,42.30,1.440980,17.291765,Maintain customer relationship through loyalty...
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,0.648511,64.851097,1,Churn,Medium Risk,2,141.40,45.849727,550.196726,"Monitor customer engagement, send personalized..."
